# Combine Chromosomes

In [4]:
%%bash

#!/bin/bash

set -o pipefail
set -o errexit

# Grab existing run info
source "/home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/latest_run.env"

echo "Combining outputs from ${OUTPUT_PATH}"

PGS_TAG="${PGS// /_}"
log_file="${OUTPUT_PATH}/combine_chr_${PGS_TAG}_${RUN_ID}_log.txt"
exec > >(tee -a "${log_file}") 2>&1

read -r -a PGS_IDs <<< "${PGS}"

for PGS_ID in "${PGS_IDs[@]}"; do
    echo "Running ${PGS_ID}"
    combined_file="${OUTPUT_PATH}/${PGS_ID}_combined_chr1_22_${RUN_ID}.csv"
    contributions_file="${OUTPUT_PATH}/${PGS_ID}_chr_contributions_${RUN_ID}.tsv"
    : > "${contributions_file}"

    for chromo in {1..22} X; do
        sscore_file="${OUTPUT_PATH}/scores_chr${chromo}.sscore"
        score_list_file="${OUTPUT_PATH}/score_list_chr${chromo}.txt"
        score_subset="${OUTPUT_PATH}/${PGS_ID}_chr${chromo}_score.txt"

        if [[ ! -f "${sscore_file}" || ! -f "${score_list_file}" ]]; then
            echo "Missing score output or score list for chr${chromo}; skipping."
            continue
        fi

        # The score column number follows this chromosome's score-list order.
        score_idx=$(awk -v target="${score_subset}" \
            '$0 == target { print NR; exit }' "${score_list_file}")
        
        # This PGS had no variants on this chromosome, so its sum contribution is zero.
        if [[ -z "${score_idx}" ]]; then
            echo "${PGS_ID}: no score file for chr${chromo}; contribution is zero."
            continue
        fi

        # Extract this PGS's raw score sum, keyed by family and individual ID.
        awk -F '\t' -v idx="${score_idx}" '
            BEGIN { OFS = "\t" }
            FNR == 1 {
                for (i = 1; i <= NF; i++) {
                    name = $i
                    sub(/^#/, "", name)
                    if (name == "FID") fid_col = i
                    if (name == "IID") iid_col = i
                    if (name == "SCORE" idx "_AVG") score_col = i
                }
                if (!fid_col || !iid_col || !score_col) {
                    print "Required FID, IID, or SCORE" idx "_AVG column missing" > "/dev/stderr"
                    exit 2
                }
                next
            }
            { print $fid_col, $iid_col, $score_col }
        ' "${sscore_file}" >> "${contributions_file}"
    done

    # Sum per-chromosome raw scores for each sample.
    awk -F '\t' '
        BEGIN {
            OFS = ","
        }
        {
            key = $1 SUBSEP $2
            fid[key] = $1
            iid[key] = $2
            score[key] += $3
        }
        END {
            print "FID", "IID", "SCORE_AVG"
            for (key in score)
                print fid[key], iid[key], score[key]
        }
    ' "${contributions_file}" > "${combined_file}"

    echo "Combined file written to ${combined_file}"
done

Combining outputs from /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/PGS000662_PGS000004_PGS003446_PGS002727_PGS002308_PGS002810_GRCh38_20261109_144222
Running PGS000662
Missing score output or score list for chr1; skipping.
Missing score output or score list for chr2; skipping.
Missing score output or score list for chr3; skipping.
Missing score output or score list for chr4; skipping.
Missing score output or score list for chr5; skipping.
Missing score output or score list for chr6; skipping.
Missing score output or score list for chrX; skipping.
Combined file written to /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/PGS000662_PGS000004_PGS003446_PGS002727_PGS002308_PGS002810_GRCh38_20261109_144222/PGS000662_combined_chr1_22_20261109_144222.csv
Running PGS000004
Missing score output or score list for chr1; skipping.
Missing score output or score list for chr2; skipping.
Missing score output or score list for chr3; skipping.
Missing score o

# Check QC logs

In [6]:
%%bash

#!/bin/bash

set -o pipefail
set -o errexit

source "/home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/latest_run.env"

BASE_DIRECTORY="/home/jupyter/workspace/workspace-bucket/calculate_pgs"
WEIGHTS_PATH="${BASE_DIRECTORY}/pgs_bim_matched_weights"

echo "Reading PLINK logs from ${OUTPUT_PATH}"

read -r -a PGS_IDs <<< "${PGS}"

for PGS_ID in "${PGS_IDs[@]}"; do
    echo "${PGS_ID}"
    summary_csv="${OUTPUT_PATH}/variants_incorporated_by_chr_${PGS_ID}_${RUN_ID}.csv"
    echo "chromosome,valid_predictors_loaded,log_file" > "${summary_csv}"

    shopt -s nullglob

    score_files=( "${WEIGHTS_PATH}/${PGS_ID}"_plink_score_"${BUILD}"_prepared.txt )

    if (( ${#score_files[@]} == 0 )); then
        echo "ERROR: No prepared score file found in ${WEIGHTS_PATH}"
        echo "Looked for: ${WEIGHTS_PATH}/${PGS}*plink_score*${BUILD}_prepared.txt"
        exit 1
    fi

    score_file="${score_files[0]}"
    total_possible="$(wc -l < "${score_file}")"

    echo "Prepared score file: ${score_file}"
    echo "Total possible matched variants: ${total_possible}"

    declare -A processed_chromosomes
    total_predictors=0

    for log_file in "${OUTPUT_PATH}"/plink*.log; do
        base_name="$(basename "${log_file}")"

        if [[ "${base_name}" =~ chr([0-9]+|X|Y)\.log$ ]]; then
            chromosome="chr${BASH_REMATCH[1]}"
        else
            echo "Chromosome not found in filename: ${base_name}"
            continue
        fi

        if [[ -n "${processed_chromosomes[$chromosome]:-}" ]]; then
            echo "Already processed ${chromosome}; skipping ${base_name}"
            continue
        fi

         # The score column number follows this chromosome's score-list order.
        score_list_file="${OUTPUT_PATH}/score_list_${chromosome}.txt"
        score_subset="${OUTPUT_PATH}/${PGS_ID}_${chromosome}_score.txt"
        score_idx=$(awk -v target="${score_subset}" \
            '$0 == target { print NR; exit }' "${score_list_file}")

        processed_chromosomes["${chromosome}"]=1

        predictor_count="$(
            awk -v idx="${score_idx}" '
                $0 ~ "--score-list file " idx "/" {
                    if (match($0, /[0-9]+ variants processed/)) {
                        count = substr($0, RSTART, RLENGTH)
                        sub(/ variants processed/, "", count)
                        result = count
                    }
                }
                END {
                    if (result != "")
                        print result
                    else
                        print 0
                }
            ' "${log_file}"
        )"

        total_predictors=$((total_predictors + predictor_count))

        echo "${chromosome}: ${predictor_count} variants processed"
        echo "${chromosome},${predictor_count},${base_name}" >> "${summary_csv}"
    done

    echo "TOTAL valid predictors loaded: ${total_predictors}"
    echo "TOTAL,${total_predictors},all_logs" >> "${summary_csv}"
    echo "TOTAL_POSSIBLE_MATCHED_VARIANTS,${total_possible},${score_file##*/}" >> "${summary_csv}"

    percent_incorporated="$(
        awk -v used="${total_predictors}" -v possible="${total_possible}" \
            'BEGIN {
                if (possible > 0) {
                    printf "%.2f", 100 * used / possible
                } else {
                    print "NA"
                }
            }'
    )"

    echo "PERCENT_INCORPORATED,${percent_incorporated},percent_of_possible_matched_variants" >> "${summary_csv}"
    echo "Percent incorperated: ${percent_incorporated}"

    echo "Wrote summary to ${summary_csv}"
    unset processed_chromosomes
    declare -A processed_chromosomes

done

PGS_TAG="${PGS// /_}"
percent_summary="${OUTPUT_PATH}/percent_incorporated_${PGS_TAG}_${RUN_ID}.csv"

{
    printf "metric"
    for PGS_ID in "${PGS_IDs[@]}"; do
        printf ",%s" "${PGS_ID}"
    done
    printf "\npercent_incorporated"

    for PGS_ID in "${PGS_IDs[@]}"; do
        summary_csv="${OUTPUT_PATH}/variants_incorporated_by_chr_${PGS_ID}_${RUN_ID}.csv"
        percent="$(awk -F, '$1 == "PERCENT_INCORPORATED" {print $2; exit}' "${summary_csv}")"
        printf ",%s" "${percent:-NA}"
    done
    printf "\n"
} > "${percent_summary}"

echo "Wrote percent-incorporated summary to ${percent_summary}"
echo "Percent incorporated by PGS:"
cat "${percent_summary}"

Reading PLINK logs from /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/PGS000662_PGS000004_PGS003446_PGS002727_PGS002308_PGS002810_GRCh38_20261109_144222
PGS000662
Prepared score file: /home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights/PGS000662_plink_score_GRCh38_prepared.txt
Total possible matched variants: 212
chr10: 11 variants processed
chr11: 14 variants processed
chr12: 12 variants processed
chr13: 3 variants processed
chr14: 9 variants processed
chr15: 4 variants processed
chr16: 5 variants processed
chr17: 11 variants processed
chr18: 3 variants processed
chr19: 5 variants processed
chr1: 14 variants processed
chr20: 7 variants processed
chr21: 2 variants processed
chr22: 6 variants processed
chr2: 24 variants processed
chr3: 13 variants processed
chr4: 6 variants processed
chr5: 11 variants processed
chr6: 15 variants processed
chr7: 7 variants processed
chr8: 18 variants processed
chr9: 7 variants processed
chrX: 5 variants p